# Build an Autonomous AI Agent from Scratch (Python, Beginner Friendly)
**David Oluyale | Agentic AI Hub**

In this notebook you build a **research agent**. You give it a goal, like *"compare 3 laptops under $800"*.
It then works **on its own**: it plans, searches, calculates, takes notes, and finally writes a report.

**What you will build, step by step**
1. A single AI call, so you see what a plain chatbot can and cannot do.
2. Four tools: a search over a small laptop catalog, a calculator, and read and write file.
3. The agent loop, from scratch, in about 50 lines of plain Python. No framework.
4. Planning (a to-do list the agent writes for itself) and memory (notes).
5. Guardrails: a step limit, a money budget, and human approval for risky actions.
6. An evaluation: 10 test tasks, scored with real measured results.

Everything runs offline except the AI calls. The "web search" is a small local catalog of **made-up laptops**, so your results are repeatable.

## Part 1. Setup
First we install the one library we need: `anthropic`. It lets Python talk to the AI model.
The `-q` flag just keeps the install quiet.

In [ ]:
%pip install -q anthropic

Next we load your API key. **Never paste a key into a notebook.**
On Colab, open the key icon on the left (Secrets), add a secret named `ANTHROPIC_API_KEY`, and switch on notebook access.
This cell reads it from there. On your own computer, it reads it from your environment instead.
We only print **whether** a key was found. We never print the key itself.

In [ ]:
import os
try:
    from google.colab import userdata          # only exists on Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass                                        # not on Colab: key must already be in your environment

print("API key found:", bool(os.environ.get("ANTHROPIC_API_KEY")))

Now we create the **client**. Think of it as a phone line to the model.
We pick a small, fast, cheap model, because an agent makes many calls.
We also write down the price per million tokens, so we can measure what every run really costs.

In [ ]:
import anthropic, json, time, re

client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"

# USD per 1 million tokens (input, output) for this model, from the public pricing page.
PRICE_IN, PRICE_OUT = 0.10, 0.50

def cost_usd(usage):
    """Turn the token counts the API reports into dollars."""
    return (usage.input_tokens * PRICE_IN + usage.output_tokens * PRICE_OUT) / 1_000_000

print("Model:", MODEL)

## Part 2. What makes an agent autonomous?
Let's start with a plain, single AI call. We ask a question and get one answer back.
This is what a normal chatbot does: **one question in, one answer out**.

In [ ]:
def ask(prompt):
    reply = client.messages.create(
        model=MODEL, max_tokens=400, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt}],
    )
    print(f"[{reply.usage.input_tokens} tokens in, {reply.usage.output_tokens} out, ${cost_usd(reply.usage):.6f}]")
    return reply.content[0].text

print(ask("In two sentences: what is an AI agent?"))

Now ask it something it **cannot know**: the prices in *our* laptop catalog.
The model has never seen our catalog. A plain chatbot can only guess, or say it does not know.
This is the gap an agent fills: it can **go and look things up**.

In [ ]:
print(ask("In our store catalog, which laptop under $800 has the longest battery life? Answer in one sentence."))

### The agent loop
An autonomous agent repeats five steps until the goal is done:

1. **Goal**: you say what you want.
2. **Plan / think**: the model decides the next action.
3. **Act**: it calls a tool, for example a search.
4. **Observe**: it reads the tool's result.
5. **Repeat**: it goes back to step 2, until it can give a final answer.

The model does the thinking. **Our Python code** runs the loop and the tools. That is all an agent is.

## Part 3. Tools
### The data: our "web"
Our agent will search a small catalog of 24 **made-up** laptops. Each one has a price, memory (RAM), storage, battery life, weight and a rating.
This cell loads the catalog file. On Colab it downloads it from the course's GitHub repo the first time.

In [ ]:
import urllib.request, pathlib

DATA = pathlib.Path("data/laptops.json")
if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/davolu/autonomous-agent-from-scratch/main/data/laptops.json", DATA)

CATALOG = json.loads(DATA.read_text())
print(len(CATALOG), "laptops loaded")
print(json.dumps(CATALOG[0], indent=1))

Let's look at the whole catalog as a small table, so you know what the agent will be searching.
Prices are in US dollars. This is practice data, not real products.

In [ ]:
print(f"{'id':4} {'name':16} {'brand':7} {'price':>6} {'RAM':>4} {'SSD':>5} {'batt':>5} {'kg':>5} {'rate':>4}")
for p in CATALOG:
    print(f"{p['id']:4} {p['name']:16} {p['brand']:7} {p['price_usd']:>6} {p['ram_gb']:>4} "
          f"{p['storage_gb']:>5} {p['battery_hours']:>4}h {p['weight_kg']:>5} {p['rating']:>4}")

### Tool 1: search
A tool is just a **normal Python function**. This one is our fake web search.
It looks for words from the query in each laptop's name, brand, chip and tags.
It can also filter by a maximum price, minimum RAM, and brand. Results come back sorted from cheapest to most expensive.

In [ ]:
STOP_WORDS = {"laptop", "laptops", "a", "the", "for", "with", "and", "best", "cheap", "under"}

def search_products(query="", max_price=None, min_ram_gb=None, brand=None):
    words = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in STOP_WORDS]
    hits = []
    for p in CATALOG:
        text = " ".join([p["name"], p["brand"], p["cpu"], " ".join(p["tags"])]).lower()
        if words and not any(w in text for w in words):
            continue
        if max_price is not None and p["price_usd"] > max_price:
            continue
        if min_ram_gb is not None and p["ram_gb"] < min_ram_gb:
            continue
        if brand and p["brand"].lower() != brand.lower():
            continue
        hits.append(p)
    hits.sort(key=lambda p: p["price_usd"])
    return {"count": len(hits), "results": hits}

Let's test the search ourselves before we give it to the agent. First: every **gaming** laptop.

In [ ]:
r = search_products("gaming")
print(r["count"], "results")
for p in r["results"]:
    print(p["id"], p["name"], "$", p["price_usd"], "| rating", p["rating"])

Second test: **student** laptops that cost at most $700 and have at least 16 GB of RAM.
Notice how the filters narrow the list down.

In [ ]:
r = search_products("student", max_price=700, min_ram_gb=16)
print(r["count"], "results")
for p in r["results"]:
    print(p["id"], p["name"], "$", p["price_usd"], "|", p["ram_gb"], "GB RAM |", p["battery_hours"], "h battery")

Third test: an empty query with only a price filter returns **everything** under that price.
The agent will use this a lot when it needs to compare all options.

In [ ]:
r = search_products(max_price=600)
print(r["count"], "results:", [p["name"] for p in r["results"]])

### Tool 2: calculator
AI models can make small mistakes with arithmetic. So we give the agent a calculator.
We do **not** use Python's `eval`, because that would run any code. Instead we read the maths expression safely, and allow only numbers and + - * / ( ).

In [ ]:
import ast, operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}

def calculator(expression):
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.operand))
        raise ValueError("only numbers and + - * / ( ) are allowed")
    try:
        return {"result": round(walk(ast.parse(expression, mode="eval").body), 4)}
    except Exception as e:
        return {"error": str(e)}

Let's test it. A normal sum works, an average works, and a dangerous input is **refused** instead of being run.

In [ ]:
print(calculator("649 + 729 + 779"))
print(calculator("(649 + 729 + 779) / 3"))
print(calculator("__import__('os').getcwd()"))

### Tools 3 and 4: write and read files
The agent will save its final report as a file. These two tools only work inside a `workspace` folder,
so the agent cannot touch any other files on your computer.

In [ ]:
WORKSPACE = pathlib.Path("workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

def safe_path(path):
    full = (WORKSPACE / path).resolve()
    if WORKSPACE not in full.parents and full != WORKSPACE:
        raise ValueError("path is outside the workspace folder")
    return full

def write_file(path, content):
    try:
        f = safe_path(path)
        f.parent.mkdir(parents=True, exist_ok=True)
        f.write_text(content)
        return {"ok": True, "path": str(f.relative_to(WORKSPACE)), "chars": len(content)}
    except Exception as e:
        return {"error": str(e)}

def read_file(path):
    try:
        return {"content": safe_path(path).read_text()[:4000]}
    except Exception as e:
        return {"error": str(e)}

A quick test: write a file, read it back, and try to escape the workspace folder. The last one is blocked.

In [ ]:
print(write_file("hello.md", "# Hello agent"))
print(read_file("hello.md"))
print(write_file("../outside.md", "should not work"))

### Describing the tools to the model
The model cannot see our Python code. So we describe each tool in a small JSON **schema**:
its name, what it does, and which inputs it takes. Good descriptions matter a lot: this is how the model decides which tool to use.

In [ ]:
TOOLS = [
    {"name": "search_products",
     "description": "Search the laptop catalog. Words in query match name, brand, chip and tags "
                    "(e.g. gaming, student, office, light, creator, budget). Empty query = all laptops. "
                    "Results are sorted by price, cheapest first, with full specs.",
     "input_schema": {"type": "object", "properties": {
         "query": {"type": "string"},
         "max_price": {"type": "number", "description": "maximum price in USD"},
         "min_ram_gb": {"type": "integer"},
         "brand": {"type": "string"}}}},
    {"name": "calculator",
     "description": "Do exact arithmetic. Input like '(649 + 729) / 2'.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}},
                      "required": ["expression"]}},
    {"name": "write_file",
     "description": "Save text to a file in the workspace, e.g. reports/laptops.md",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
                      "required": ["path", "content"]}},
    {"name": "read_file",
     "description": "Read a text file from the workspace.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}},
]
print([t["name"] for t in TOOLS])

Finally, a small **dispatcher**. When the model asks for a tool by name, this looks up the matching Python function,
runs it with the model's inputs, and turns the result into text the model can read.

In [ ]:
TOOL_FUNCS = {"search_products": search_products, "calculator": calculator,
              "write_file": write_file, "read_file": read_file}

def run_tool(name, args):
    if name not in TOOL_FUNCS:
        return json.dumps({"error": f"unknown tool {name}"})
    try:
        return json.dumps(TOOL_FUNCS[name](**args))
    except Exception as e:
        return json.dumps({"error": str(e)})

print(run_tool("calculator", {"expression": "799 - 649"}))

## Part 4. The core loop, from scratch
This is the heart of the course. Read it slowly. It is the five steps from Part 2, in code:

- `messages` is the conversation so far. It starts with the **goal**.
- Each turn of the `for` loop, the model **thinks** and either asks for tools or gives a final answer.
- If it asks for tools, we **act** (run them) and send back the results so it can **observe**.
- Then we **repeat**. A `max_steps` limit stops it from looping forever.

In [ ]:
SYSTEM = """You are a careful research agent. You work on your own until the goal is done.
Always use the tools to find facts. Never guess prices or specs.
Use the calculator for any arithmetic. When finished, give a short, clear answer."""

def short(text, n=160):
    text = str(text).replace("\n", " ")
    return text if len(text) <= n else text[:n] + " ..."

def run_agent(goal, tools=TOOLS, system=SYSTEM, max_steps=10, verbose=True):
    messages = [{"role": "user", "content": goal}]                 # 1. GOAL
    trace, spent = [], 0.0
    for step in range(1, max_steps + 1):
        reply = client.messages.create(                             # 2. THINK / PLAN
            model=MODEL, max_tokens=2000, system=system, tools=tools,
            messages=messages, thinking={"type": "disabled"})
        spent += cost_usd(reply.usage)
        messages.append({"role": "assistant", "content": reply.content})

        if reply.stop_reason != "tool_use":                         # no tool asked: done
            answer = "".join(b.text for b in reply.content if b.type == "text")
            return {"answer": answer, "steps": step, "trace": trace, "usd": spent, "stopped": "done"}

        results = []
        for block in reply.content:
            if block.type == "tool_use":
                output = run_tool(block.name, block.input)          # 3. ACT
                trace.append((step, block.name, block.input))
                if verbose:
                    print(f"step {step}: {block.name}({short(json.dumps(block.input), 90)})")
                    print(f"        -> {short(output)}")
                results.append({"type": "tool_result",              # 4. OBSERVE
                                "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})       # 5. REPEAT
    return {"answer": None, "steps": max_steps, "trace": trace, "usd": spent, "stopped": "max_steps"}

### First run
Let's give the agent its first goal. Watch the printed steps: each line is one tool call the agent decided to make **by itself**.
We never told it which tools to use, or in what order.

In [ ]:
t0 = time.time()
result = run_agent("Compare 3 laptops under $800 that are good for students. Which one would you pick and why?")
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f} | time: {time.time() - t0:.1f}s")

### Reading the trace
The `trace` is a record of every action the agent took. This is how you **debug** an agent:
you look at what it did, step by step, not only at its final answer.

In [ ]:
for step, name, args in result["trace"]:
    print(f"step {step} -> {name}: {args}")

### A second goal that needs maths
This goal needs the calculator. The agent has to find prices, then work out a difference.
Notice it uses **more than one tool** to finish.

In [ ]:
result = run_agent("What is the price difference between the cheapest and the most expensive Forge laptop? "
                   "Use the calculator for the difference.")
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f}")

## Part 5. Planning and memory
On bigger goals, agents can lose track. Two simple ideas help a lot:

- **Planning**: the agent writes its own to-do list first, and ticks items off as it goes.
- **Memory**: the agent saves short notes about important facts, so it does not forget them later.

Both are just more tools. They save into a Python dictionary called `MEMORY`.

In [ ]:
MEMORY = {"plan": [], "notes": []}

def update_plan(items):
    """Replace the to-do list. items = [{"task": "...", "done": false}, ...]"""
    MEMORY["plan"] = items
    return {"plan": [("[x] " if i.get("done") else "[ ] ") + i["task"] for i in items]}

def save_note(note):
    MEMORY["notes"].append(note)
    return {"saved": True, "notes_so_far": len(MEMORY["notes"])}

def read_notes():
    return {"notes": MEMORY["notes"]}

TOOL_FUNCS.update({"update_plan": update_plan, "save_note": save_note, "read_notes": read_notes})
print(update_plan([{"task": "test the plan tool", "done": True}]))
MEMORY = {"plan": [], "notes": []}

We describe the three new tools to the model, just like before, and add them to the tool list.

In [ ]:
MEMORY_TOOLS = [
    {"name": "update_plan",
     "description": "Write or update your to-do list. Call it first with all steps, then again as you finish them.",
     "input_schema": {"type": "object", "properties": {"items": {"type": "array", "items": {
         "type": "object", "properties": {"task": {"type": "string"}, "done": {"type": "boolean"}},
         "required": ["task", "done"]}}}, "required": ["items"]}},
    {"name": "save_note",
     "description": "Save one short fact you found (e.g. 'L07 Slate 14: $729, 13h battery').",
     "input_schema": {"type": "object", "properties": {"note": {"type": "string"}}, "required": ["note"]}},
    {"name": "read_notes",
     "description": "Read all notes you saved so far.",
     "input_schema": {"type": "object", "properties": {}}},
]
PLANNER_TOOLS = TOOLS + MEMORY_TOOLS
print([t["name"] for t in PLANNER_TOOLS])

The system prompt now tells the agent **how to work**: plan first, take notes, read the notes before writing,
and save the final report as a file. The loop itself does not change at all.

In [ ]:
PLANNER_SYSTEM = SYSTEM + """
How to work:
1. First call update_plan with your full to-do list.
2. Save each important fact with save_note.
3. Mark plan items done with update_plan as you finish them.
4. Before writing, call read_notes. Then save a markdown report with write_file in reports/.
5. Finish with a short summary."""
print(PLANNER_SYSTEM)

### Running the planning agent
Now a bigger goal: research, compare, and write a report. Watch for the plan at the start, the notes in the middle, and the file at the end.

In [ ]:
MEMORY = {"plan": [], "notes": []}
t0 = time.time()
result = run_agent("Research the 3 best laptops under $800 for a university student who travels a lot "
                   "(battery life and weight matter). Write a comparison report to reports/student_laptops.md.",
                   tools=PLANNER_TOOLS, system=PLANNER_SYSTEM, max_steps=20)
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f} | time: {time.time() - t0:.1f}s")

Let's look inside the agent's memory. First its **plan**, the to-do list it wrote for itself, as it was at the end.

In [ ]:
for item in MEMORY["plan"]:
    print(("[x] " if item.get("done") else "[ ] ") + item["task"])

And here are the **notes** it saved while it was working.

In [ ]:
for i, note in enumerate(MEMORY["notes"], 1):
    print(f"{i}. {note}")

Finally, the **report** it wrote. This file was created by the agent, not by us.

In [ ]:
print((WORKSPACE / "reports/student_laptops.md").read_text())

## Part 6. Guardrails
An agent that acts on its own needs limits. We add three:

1. **Max steps**: stop after N steps, so it can never loop forever.
2. **Budget cap**: stop when the money spent reaches a limit.
3. **Human approval**: risky tools (here, writing files) need a "yes" first.

First, the approval rule. In `policy` mode it approves writes only to markdown files in `reports/`.
In `ask` mode it asks you, the human, by typing y or n. Every decision is printed, so you can audit it.

In [ ]:
RISKY_TOOLS = {"write_file"}
APPROVAL_MODE = "policy"        # change to "ask" to approve each risky action yourself

def approve(name, args):
    if name not in RISKY_TOOLS:
        return True
    if APPROVAL_MODE == "ask":
        return input(f"Allow {name} {args.get('path')}? (y/n) ").strip().lower() == "y"
    path = str(args.get("path", ""))
    return path.startswith("reports/") and path.endswith(".md") and ".." not in path

print(approve("calculator", {"expression": "1+1"}))
print(approve("write_file", {"path": "reports/ok.md"}))
print(approve("write_file", {"path": "notes/secret.txt"}))

Now the **safe agent**. It is the same loop as before, with three additions, marked with comments:
a budget check before each model call, an approval check before each risky tool, and a clear reason when it stops.

In [ ]:
def run_agent_safe(goal, tools=PLANNER_TOOLS, system=PLANNER_SYSTEM, max_steps=15, budget_usd=0.02, verbose=True):
    messages = [{"role": "user", "content": goal}]
    trace, spent = [], 0.0
    for step in range(1, max_steps + 1):
        if spent >= budget_usd:                                          # GUARDRAIL 2: budget
            return {"answer": None, "steps": step - 1, "trace": trace, "usd": spent, "stopped": "budget"}
        reply = client.messages.create(model=MODEL, max_tokens=2000, system=system, tools=tools,
                                       messages=messages, thinking={"type": "disabled"})
        spent += cost_usd(reply.usage)
        messages.append({"role": "assistant", "content": reply.content})
        if reply.stop_reason != "tool_use":
            answer = "".join(b.text for b in reply.content if b.type == "text")
            return {"answer": answer, "steps": step, "trace": trace, "usd": spent, "stopped": "done"}
        results = []
        for block in reply.content:
            if block.type != "tool_use":
                continue
            if approve(block.name, block.input):                         # GUARDRAIL 3: approval
                output = run_tool(block.name, block.input)
            else:
                output = json.dumps({"error": "DENIED by human approval policy. "
                                              "Only markdown files in reports/ may be written."})
            trace.append((step, block.name, block.input))
            if verbose:
                print(f"step {step}: {block.name}({short(json.dumps(block.input), 90)})")
                print(f"        -> {short(output)}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})
    return {"answer": None, "steps": max_steps, "trace": trace, "usd": spent, "stopped": "max_steps"}  # GUARDRAIL 1

### Guardrail test 1: max steps
We give a big goal but allow only **2 steps**. The agent must stop early, and it tells us why.

In [ ]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Write a full report on every laptop brand in the catalog.", max_steps=2)
print("\nstopped because:", r["stopped"], "| steps:", r["steps"], f"| cost: ${r['usd']:.5f}")

### Guardrail test 2: budget cap
Now we set a tiny budget: one tenth of a cent. The agent stops as soon as it has spent that much.

In [ ]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Write a full report on every laptop brand in the catalog.", budget_usd=0.001)
print("\nstopped because:", r["stopped"], "| steps:", r["steps"], f"| cost: ${r['usd']:.5f}")

### Guardrail test 3: human approval
We ask the agent to save a file in a place that is **not allowed**. The approval rule denies it,
and the agent receives that "denied" message as an observation, so it can adapt.

In [ ]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Find the cheapest gaming laptop and save a one-line summary to notes/gaming.txt.",
                   max_steps=10)
print("\n=== ANSWER ===")
print(r["answer"])
print("\nstopped because:", r["stopped"], f"| cost: ${r['usd']:.5f}")
print("files in workspace:", sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file()))

### Worked example: a full guarded run
Let's use the finished, safe agent on a realistic request. It plans, searches, takes notes, does maths,
and writes a report, all inside its limits.

In [ ]:
MEMORY = {"plan": [], "notes": []}
t0 = time.time()
r = run_agent_safe("I edit videos and have $1,200. Find the 3 best laptops for content creators within budget. "
                   "Include the average price of the three (use the calculator). Save the report to reports/creator.md.",
                   max_steps=20, budget_usd=0.05)
print("\n=== ANSWER ===")
print(r["answer"])
print(f"\nstopped: {r['stopped']} | steps: {r['steps']} | cost: ${r['usd']:.5f} | time: {time.time() - t0:.1f}s")

Here is the creator report the agent saved.

In [ ]:
print((WORKSPACE / "reports/creator.md").read_text())

## Part 7. Evaluation
How do we know the agent is any good? We **test it**, like any other software.
We write 10 tasks. For each one, we compute the correct answer **ourselves, with plain Python** from the catalog.
Then we compare the agent's answer to it. No opinions, just measured results.

In [ ]:
C = CATALOG
def cheapest(rows): return min(rows, key=lambda p: p["price_usd"])

TASKS = [
    ("Which laptop is the cheapest in the whole catalog?", cheapest(C)["id"]),
    ("Which is the cheapest laptop with at least 16 GB of RAM?", cheapest([p for p in C if p["ram_gb"] >= 16])["id"]),
    ("Which laptop under $800 has the longest battery life?",
        max([p for p in C if p["price_usd"] < 800], key=lambda p: p["battery_hours"])["id"]),
    ("Which is the lightest laptop under $1000?",
        min([p for p in C if p["price_usd"] < 1000], key=lambda p: p["weight_kg"])["id"]),
    ("Which gaming laptop has the highest rating?", max([p for p in C if "gaming" in p["tags"]], key=lambda p: p["rating"])["id"]),
    ("Which is the cheapest laptop with at least 1 TB (1024 GB) of storage?",
        cheapest([p for p in C if p["storage_gb"] >= 1024])["id"]),
    ("Which laptop under $600 has the best rating?", max([p for p in C if p["price_usd"] < 600], key=lambda p: p["rating"])["id"]),
    ("What is the total price in USD of the two cheapest laptops?", sum(sorted(p["price_usd"] for p in C)[:2])),
    ("What is the average price in USD of all Forge laptops?",
        round(sum(p["price_usd"] for p in C if p["brand"] == "Forge") / len([p for p in C if p["brand"] == "Forge"]), 2)),
    ("How many laptops cost under $700 and have at least 16 GB of RAM?",
        len([p for p in C if p["price_usd"] < 700 and p["ram_gb"] >= 16])),
]
for i, (q, a) in enumerate(TASKS, 1):
    print(f"{i:2}. {q}  ->  expected: {a}")

The agent must end its answer with a line like `FINAL ANSWER: L07` or `FINAL ANSWER: 658`.
This **checker** reads that line and compares it with the expected answer. Numbers may differ by at most 0.01.

In [ ]:
EVAL_SYSTEM = SYSTEM + "\nEnd your reply with one line: FINAL ANSWER: <laptop id like L07, or a number>"

def check(answer, expected):
    m = re.search(r"FINAL ANSWER:\s*\$?([A-Za-z0-9.,]+)", answer or "")
    if not m:
        return False, None
    got = m.group(1).rstrip(".,").replace(",", "")
    if isinstance(expected, str):
        return got.upper() == expected, got
    try:
        return abs(float(got) - expected) <= 0.01, got
    except ValueError:
        return False, got

print(check("blah\nFINAL ANSWER: L15", "L15"), check("FINAL ANSWER: $1,129.00", 1129.0), check("no idea", 2))

Now we run all 10 tasks. For each one we record: pass or fail, the agent's answer, the steps it took, the cost and the time.
This takes a minute or two, because the agent works through each task on its own.

In [ ]:
rows = []
for i, (q, expected) in enumerate(TASKS, 1):
    t0 = time.time()
    r = run_agent_safe(q, tools=TOOLS, system=EVAL_SYSTEM, max_steps=8, budget_usd=0.02, verbose=False)
    ok, got = check(r["answer"], expected)
    rows.append({"task": i, "pass": ok, "got": got, "expected": expected, "steps": r["steps"],
                 "usd": round(r["usd"], 5), "sec": round(time.time() - t0, 1), "stopped": r["stopped"]})
    print(f"task {i:2}: {'PASS' if ok else 'FAIL'}  got={got}  expected={expected}  "
          f"steps={r['steps']}  ${r['usd']:.5f}  {rows[-1]['sec']}s")

### The score
Here is the summary, measured from the run above: how many tasks passed, the total cost, and the average steps and time per task.

In [ ]:
passed = sum(r["pass"] for r in rows)
print(f"Score: {passed}/{len(rows)} tasks passed")
print(f"Total cost: ${sum(r['usd'] for r in rows):.5f}")
print(f"Average steps per task: {sum(r['steps'] for r in rows) / len(rows):.1f}")
print(f"Average time per task: {sum(r['sec'] for r in rows) / len(rows):.1f}s")
pathlib.Path("results").mkdir(exist_ok=True)
json.dump(rows, open("results/eval_results.json", "w"), indent=1)
failed = [r for r in rows if not r["pass"]]
print("Failed tasks:", [(r["task"], r["got"], r["expected"]) for r in failed] or "none")

## Part 8. Recap and next steps
You built a real autonomous agent from scratch:

- **The loop**: goal, think, act, observe, repeat, in about 50 lines of Python.
- **Tools**: search, calculator, read and write file, each a plain function plus a short description.
- **Planning and memory**: a to-do list and notes, added as more tools.
- **Guardrails**: max steps, a budget cap, and human approval for risky actions.
- **Evaluation**: 10 tasks with answers we computed ourselves, scored automatically.

**Try next**
- Swap `search_products` for a real web search API.
- Add your own tool, for example sending a draft email that always needs approval.
- Add harder tasks to the evaluation, and re-run it every time you change the prompt.

*David Oluyale | Agentic AI Hub*